In [ ]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 11.1 MB/s eta 0:00:00


In [ ]:
import re
import pandas as pd
from IPython.display import display
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

stopwords = set(StopWordRemoverFactory().get_stop_words())
stemmer = StemmerFactory().create_stemmer()

print(f"Jumlah stopwords Sastrawi: {len(stopwords)}")

Jumlah stopwords Sastrawi: 123


In [ ]:
dokumen = [
    {"ID": "D1", "Teks Asli": "Mahasiswa mempelajari sistem komputer dan jaringan untuk memahami teknologi informasi."},
    {"ID": "D2", "Teks Asli": "Jaringan komputer menghubungkan perangkat agar pengguna dapat berbagi data dengan cepat."},
    {"ID": "D3", "Teks Asli": "Keamanan sistem informasi penting untuk melindungi data pengguna dari serangan siber."},
    {"ID": "D4", "Teks Asli": "Teknologi kecerdasan buatan membantu komputer menganalisis data dan membuat keputusan."},
    {"ID": "D5", "Teks Asli": "Pengembang perangkat lunak menguji aplikasi agar sistem berjalan dengan baik."}
]
df_dokumen = pd.DataFrame(dokumen)
display(df_dokumen)

,ID,Teks Asli
0,D1,Mahasiswa mempelajari sistem komputer dan jari...
1,D2,Jaringan komputer menghubungkan perangkat agar...
2,D3,Keamanan sistem informasi penting untuk melind...
3,D4,Teknologi kecerdasan buatan membantu komputer ...
4,D5,Pengembang perangkat lunak menguji aplikasi ag...


In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\\s]", " ", text)
    tokens = text.split()
    tokens = [token for token in tokens if token not in stopwords]
    tokens = [stemmer.stem(token) for token in tokens]
    return tokens

def token_sebelum_preprocessing(text):
    text = text.lower()
    text = re.sub(r"[^a-z\\s]", " ", text)
    return text.split()

contoh = "Mahasiswa MEMPELAJARI sistem komputer!"
print("Hasil preprocessing:", preprocess_text(contoh))

Hasil preprocessing: ['mahasiswa', 'ajar', 'sistem', 'komputer']


In [ ]:
hasil = []

for item in dokumen:
    sebelum = token_sebelum_preprocessing(item["Teks Asli"])
    sesudah = preprocess_text(item["Teks Asli"])

    hasil.append({
        "ID": item["ID"],
        "Teks Asli": item["Teks Asli"],
        "Token Sebelum": " ".join(sebelum),
        "Token Sesudah": " ".join(sesudah),
        "Jumlah Token Sebelum": len(sebelum),
        "Jumlah Token Sesudah": len(sesudah)
    })

df_hasil = pd.DataFrame(hasil)
display(df_hasil)

,ID,Teks Asli,Token Sebelum,Token Sesudah,Jumlah Token Sebelum,Jumlah Token Sesudah
0,D1,Mahasiswa mempelajari sistem komputer dan jari...,mahasiswa mempelajari sistem komputer dan jari...,mahasiswa ajar sistem komputer jaring paham te...,10,8
1,D2,Jaringan komputer menghubungkan perangkat agar...,jaringan komputer menghubungkan perangkat agar...,jaring komputer hubung perangkat guna bagi dat...,11,8
2,D3,Keamanan sistem informasi penting untuk melind...,keamanan sistem informasi penting untuk melind...,aman sistem informasi penting lindung data gun...,11,9
3,D4,Teknologi kecerdasan buatan membantu komputer ...,teknologi kecerdasan buatan membantu komputer ...,teknologi cerdas buat bantu komputer analis da...,10,9
4,D5,Pengembang perangkat lunak menguji aplikasi ag...,pengembang perangkat lunak menguji aplikasi ag...,kembang perangkat lunak uji aplikasi sistem ja...,10,8


In [ ]:
df_hasil["Pengurangan Token (%)"] = df_hasil.apply(
    lambda r: ((r["Jumlah Token Sebelum"] - r["Jumlah Token Sesudah"])
               / r["Jumlah Token Sebelum"] * 100)
    if r["Jumlah Token Sebelum"] else 0,
    axis=1
).round(2)

total_sebelum = int(df_hasil["Jumlah Token Sebelum"].sum())
total_sesudah = int(df_hasil["Jumlah Token Sesudah"].sum())
pengurangan_total = ((total_sebelum - total_sesudah) / total_sebelum * 100) if total_sebelum else 0

print("Rincian per dokumen:")
display(df_hasil[["ID", "Jumlah Token Sebelum", "Jumlah Token Sesudah", "Pengurangan Token (%)"]])

print("Ringkasan seluruh dokumen:")
display(pd.DataFrame({
    "Keterangan": ["Jumlah token sebelum preprocessing", "Jumlah token sesudah preprocessing", "Persentase pengurangan token"],
    "Hasil": [total_sebelum, total_sesudah, round(pengurangan_total, 2)]
}))

Rincian per dokumen:


,ID,Jumlah Token Sebelum,Jumlah Token Sesudah,Pengurangan Token (%)
0,D1,10,8,20.00
1,D2,11,8,27.27
2,D3,11,9,18.18
3,D4,10,9,10.00
4,D5,10,8,20.00


Ringkasan seluruh dokumen:


,Keterangan,Hasil
0,Jumlah token sebelum preprocessing,52.00
1,Jumlah token sesudah preprocessing,42.00
2,Persentase pengurangan token,19.23


Analisis Singkat:
Tahap preprocessing digunakan untuk mempersiapkan data teks agar lebih mudah diolah oleh sistem. Case folding membuat penulisan huruf menjadi seragam, sedangkan cleaning menghilangkan karakter atau simbol yang tidak diperlukan. Stopwords removal menyaring kata-kata yang kurang memberikan informasi penting, sementara stemming mengubah kata berimbuhan menjadi bentuk dasarnya. Proses ini membuat data teks lebih terstruktur dan membantu meningkatkan efektivitas proses pencarian. Namun, setiap tahap perlu dilakukan dengan tepat agar informasi penting dalam teks tidak ikut terhapus.